# Option pool assembly: join decisions with real options and proposals
Join 45,500 decisions with hybrid real options (T1/T2/T3) and Qwen proposals.

**Join rule:** (source, instance_id, step)  
**Output:** rows_proposal.jsonl.gz with full option pools  
**Soft targets:** gold=1.0, T1=0.02, T2=0.10, T3=0.25, proposal=0.5, NONE=0.0; renormalized

In [ ]:
import json, gzip, os
from collections import Counter, defaultdict

print("Loading 45,500 decisions...")
decisions = {}
input_dir = "/kaggle/input/laya-head-states-v1"
count = 0
if os.path.exists(input_dir):
    for shard_file in sorted([f for f in os.listdir(input_dir) if f.startswith("pool_") and f.endswith(".jsonl")]):
        with open(os.path.join(input_dir, shard_file)) as f:
            for line in f:
                r = json.loads(line)
                decisions[r.get("decision_id", "")] = r
                count += 1
else:
    # Dummy data for smoke
    decisions = {f"d{i}": {"decision_id": f"d{i}", "source": "test", "instance_id": f"inst{i}", "state_ref": {"step": 0}, "options": [{"text": f"action {i}"}]} for i in range(10)}

print(f"Loaded {len(decisions)} decisions (attempted {count})")

In [ ]:
print("\nLoading hybrid real options...")
hybrid_by_key = defaultdict(list)

# Try to load from Kaggle dataset (omitted in smoke; would load rows_hybrid.jsonl.gz)
hybrid_dir = "/kaggle/input/hybrid-options"
if os.path.exists(hybrid_dir):
    try:
        with gzip.open(os.path.join(hybrid_dir, "rows_hybrid.jsonl.gz")) as f:
            for line in f:
                row = json.loads(line)
                key = (row.get("source"), row.get("instance_id"), row.get("state_ref", {}).get("step"))
                hybrid_by_key[key].append(row)
    except FileNotFoundError:
        print("rows_hybrid.jsonl.gz not found")
else:
    print("Hybrid dataset not available (expected before full run)")

print(f"Loaded {sum(len(v) for v in hybrid_by_key.values())} hybrid options from {len(hybrid_by_key)} keys")

In [ ]:
print("\nLoading proposals...")
proposals_by_id = {}

for shard in [0, 1]:
    path = f"/kaggle/working/out/proposals_shard{shard}.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                proposals_by_id[r["decision_id"]] = r
    else:
        print(f"Shard {shard} not yet available (expected in full run)")

print(f"Loaded {len(proposals_by_id)} proposals")

In [ ]:
print("\nAssembling option pools...")
rows_out = []
join_count = 0
proposal_count = 0
hit_count = 0

for i, (decision_id, dec) in enumerate(decisions.items()):
    if i % 5000 == 0:
        print(f"  {i}/{len(decisions)}...")
    
    gold = dec.get("options", [{}])[0]
    gold_text = gold.get("text", "")
    
    # Join on (source, instance_id, step)
    source = dec.get("source", "unknown")
    instance_id = dec.get("instance_id", "unknown")
    step = dec.get("state_ref", {}).get("step", 0)
    key = (source, instance_id, step)
    
    real_options = hybrid_by_key.get(key, [])
    if real_options:
        join_count += 1
    
    # Get proposal
    proposal = proposals_by_id.get(decision_id, {})
    
    # Compile option list: gold + real + proposal + NONE
    options_out = []
    soft_targets_raw = []
    
    # Gold
    options_out.append({
        "text": gold_text,
        "tier": "GOLD",
        "provenance": gold.get("provenance", {}),
    })
    soft_targets_raw.append(1.0)
    
    # Real options (T1/T2/T3)
    for real_opt in real_options:
        opt_tier = real_opt.get("tier", "T3")
        soft_target = {"T1": 0.02, "T2": 0.10, "T3": 0.25}.get(opt_tier, 0.25)
        options_out.append({
            "text": real_opt.get("text", ""),
            "tier": opt_tier,
            "provenance": real_opt.get("provenance", {}),
        })
        soft_targets_raw.append(soft_target)
    
    # Proposal
    if proposal.get("proposal_text"):
        proposal_count += 1
        is_hit = proposal["proposal_text"].strip() == gold_text.strip()
        if is_hit:
            hit_count += 1
        options_out.append({
            "text": proposal["proposal_text"],
            "tier": "proposal",
            "provenance": {"proposal_id": proposal.get("proposal_id")},
            "is_gold_hit": is_hit,
        })
        soft_targets_raw.append(0.5)
    
    # NONE
    options_out.append({
        "text": "NONE",
        "tier": "NONE",
    })
    soft_targets_raw.append(0.0)
    
    # Normalize soft targets
    total = sum(soft_targets_raw)
    soft_targets = [s / total for s in soft_targets_raw]
    for opt, st in zip(options_out, soft_targets):
        opt["soft_target"] = st
    
    row = {
        "decision_id": decision_id,
        "state_ref": dec.get("state_ref", {}),
        "options": options_out,
    }
    rows_out.append(row)

print(f"\nAssembled {len(rows_out)} decisions")
print(f"  Joined with real options: {join_count}")
print(f"  Proposals added: {proposal_count}")
print(f"  Proposals matching gold: {hit_count}")

In [ ]:
print("\nSaving output...")
os.makedirs("/kaggle/working/out", exist_ok=True)

# Write rows
with gzip.open("/kaggle/working/out/rows_proposal.jsonl.gz", "wt", encoding="utf-8") as f:
    for row in rows_out:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Write manifest
manifest = {
    "label": "proposal pool assembly",
    "date": "2026-10-04",
    "total_rows": len(rows_out),
    "rows_with_real_options": join_count,
    "proposal_count": proposal_count,
    "proposal_hit_rate": hit_count / max(1, proposal_count),
    "soft_target_config": {"gold": 1.0, "T1": 0.02, "T2": 0.10, "T3": 0.25, "proposal": 0.5, "NONE": 0.0},
}

with open("/kaggle/working/out/manifest_proposal.json", "w") as f:
    json.dump(manifest, f, indent=1)

print(f"\nWrote:")
print(f"  rows_proposal.jsonl.gz ({len(rows_out)} rows)")
print(f"  manifest_proposal.json")
print(f"\nManifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")